# H27 FLOW_HTBRANCHPINNTRAJ Training

Flow-based review-aligned ablation: compact `CFAST_ORANGE3` condition + internal branch embedding + PINN trajectory surrogate signal.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('not in Colab or Drive already unavailable:', repr(exc))

from pathlib import Path
import os

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/content/drive/MyDrive/Colab Notebooks/final_submission_h27_flow_main_20260623'),
    Path('/content/drive/MyDrive/final_submission_h27_flow_main_20260623'),
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'scripts').exists()), Path.cwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
PREPARED = Path('outputs/h27_context_ablation_140k_cfast_prepare/prepared_flow_pilot_data.npz')
OUT_ROOT = Path('outputs/experiments/20260622_h27_flow_htbranchpinntraj')
REFERENCE_DIR = Path('FMO_H27_context_ablation/data/clustered_from_clean/dynamic_condition_modes_n1000')

BASE_CONDITION = 'CFAST_ORANGE3'
METHOD = 'FLOW_HTBRANCHPINNTRAJ'
CONDITION_SET = f'{BASE_CONDITION}_{METHOD}'

RUN_METADATA = True
RUN_PINN_ONLY = False
RUN_SMOKE = True
RUN_FULL = True
RUN_VALIDATION_AND_DIVERSITY = True
FORCE_FULL = False

SMOKE_EPOCHS = 2
FULL_EPOCHS = 500
PINN_EPOCHS = 180
BATCH_SIZE = 256
HIDDEN = 256
FLOW_LAYERS = 8
LR = 3e-4
PATIENCE = 15
N_GENERATE = 512
SEED = 20260622

AUX_START_EPOCH = 10
AUX_WARMUP_EPOCHS = 20
FLOW_AUX_WEIGHT = 1.0
FLOW_VAL_AUX_WEIGHT = 1.0
HTBAL_PRIOR_ALPHA = 0.30
HTBAL_PRIOR_MIN = 0.05
BRANCH_KL_WEIGHT_SCALE = 1.0
GENERATE_STRATEGIES = 'median,mixture'

VALIDATION_TARGETS = 'ALL'
VALIDATION_N_PER_TARGET = -1
VALIDATION_SELECTION = 'head'
VALIDATION_DT = 0.5
VALIDATION_T_MAX = 50.0
VALIDATION_LAMBDA_REORG = 35.0
SAVE_TRAJECTORIES = True
INSTALL_QUTIP_IF_MISSING = True

THREAD_ENV = {
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
    'VECLIB_MAXIMUM_THREADS': '1',
}

assert PREPARED.exists(), f'missing prepared artifact: {PREPARED}'
assert REFERENCE_DIR.exists(), f'missing dynamic reference dir: {REFERENCE_DIR}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('prepared:', PREPARED)
print('out:', OUT_ROOT.resolve())

In [ ]:
def run(cmd, env=None):
    cmd = list(map(str, cmd))
    print('\n$ ' + ' '.join(cmd), flush=True)
    merged_env = os.environ.copy()
    merged_env.update(THREAD_ENV)
    merged_env['PYTHONUNBUFFERED'] = '1'
    if env:
        merged_env.update(env)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=merged_env)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'command failed with exit code {returncode}: {cmd}')
    return subprocess.CompletedProcess(cmd, returncode)

def find_script(name):
    candidates = [Path('scripts') / name, Path.cwd() / 'scripts' / name]
    for p in candidates:
        if p.exists():
            return p
    for root in [Path.cwd(), Path('/content/drive/MyDrive')]:
        if not root.exists():
            continue
        try:
            for p in root.rglob(name):
                if p.parent.name == 'scripts':
                    return p
        except Exception as exc:
            print('script search skipped:', root, repr(exc))
    raise FileNotFoundError(name)

TRAIN_SCRIPT = find_script('train_h27_flow_htbranch_pinntraj.py')
VALIDATE_SCRIPT = find_script('validate_h27_cfast_generated_simulator_latest.py') if (Path('scripts') / 'validate_h27_cfast_generated_simulator_latest.py').exists() else find_script('validate_h27_cfast_generated_simulator.py')
DIVERSITY_SCRIPT = find_script('analyze_h27_cfast_generated_diversity.py')
print('train:', TRAIN_SCRIPT)
print('validate:', VALIDATE_SCRIPT)
print('diversity:', DIVERSITY_SCRIPT)

run(['python3', '-m', 'py_compile', TRAIN_SCRIPT])
run(['python3', '-m', 'py_compile', VALIDATE_SCRIPT])
run(['python3', '-m', 'py_compile', DIVERSITY_SCRIPT])

if INSTALL_QUTIP_IF_MISSING:
    try:
        import qutip  # noqa
        print('qutip ok')
    except ModuleNotFoundError:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'qutip'])

In [ ]:
def train_cmd(run_name, epochs, n_generate, extra=None):
    cmd = [
        'python3', '-u', TRAIN_SCRIPT,
        '--prepared', PREPARED,
        '--out-root', OUT_ROOT,
        '--run-name', run_name,
        '--base-condition', BASE_CONDITION,
        '--methods', METHOD,
        '--epochs', epochs,
        '--pinn-epochs', PINN_EPOCHS,
        '--batch-size', BATCH_SIZE,
        '--hidden', HIDDEN,
        '--flow-layers', FLOW_LAYERS,
        '--lr', LR,
        '--patience', PATIENCE,
        '--n-generate', n_generate,
        '--aux-start-epoch', AUX_START_EPOCH,
        '--aux-warmup-epochs', AUX_WARMUP_EPOCHS,
        '--flow-aux-weight', FLOW_AUX_WEIGHT,
        '--flow-val-aux-weight', FLOW_VAL_AUX_WEIGHT,
        '--htbal-prior-alpha', HTBAL_PRIOR_ALPHA,
        '--htbal-prior-min', HTBAL_PRIOR_MIN,
        '--branch-kl-weight-scale', BRANCH_KL_WEIGHT_SCALE,
        '--htbal-reference-dir', REFERENCE_DIR,
        '--generate-strategies', GENERATE_STRATEGIES,
        '--seed', SEED,
    ]
    if extra:
        cmd += extra
    return cmd

if RUN_METADATA:
    run(train_cmd('smoke', 1, 8, ['--metadata-only', '--no-progress']))

if RUN_PINN_ONLY:
    run(train_cmd('full', 1, 8, ['--pinn-only']))

if RUN_SMOKE:
    run(train_cmd('smoke', SMOKE_EPOCHS, 16, ['--force', '--pinn-epochs', 2]))

if RUN_FULL:
    extra = ['--force'] if FORCE_FULL else []
    run(train_cmd('full', FULL_EPOCHS, N_GENERATE, extra))

In [ ]:
if RUN_VALIDATION_AND_DIVERSITY:
    run_dir = OUT_ROOT / 'full'
    generated = run_dir / f'{CONDITION_SET}_generated_samples.npz'
    assert generated.exists(), f'missing generated: {generated}'
    val_out = run_dir / f'simulator_validation_{CONDITION_SET}'
    cmd = [
        'python3', '-u', VALIDATE_SCRIPT,
        '--generated', generated,
        '--out-dir', val_out,
        '--condition-set', CONDITION_SET,
        '--targets', VALIDATION_TARGETS,
        '--n-per-target', VALIDATION_N_PER_TARGET,
        '--selection', VALIDATION_SELECTION,
        '--lambda-reorg', VALIDATION_LAMBDA_REORG,
        '--t-max', VALIDATION_T_MAX,
        '--dt', VALIDATION_DT,
        '--print-every', 20,
    ]
    if SAVE_TRAJECTORIES:
        cmd.append('--save-trajectories')
    run(cmd)

    detail = val_out / 'csv' / f'{generated.stem}_simulator_validation_detail.csv'
    div_out = run_dir / f'diversity_audit_{CONDITION_SET}'
    run([
        'python3', '-u', DIVERSITY_SCRIPT,
        '--generated', generated,
        '--validation-detail', detail,
        '--prepared', PREPARED,
        '--out-dir', div_out,
        '--condition-set', CONDITION_SET,
        '--targets', 'ALL',
        '--seed', SEED,
    ])
    print('validation summary:', val_out / 'csv' / f'{generated.stem}_simulator_validation_summary.csv')
    print('diversity decision:', div_out / 'csv' / 'diversity_decision_summary.csv')

In [ ]:
import pandas as pd

paths = {
    'branch_summary': OUT_ROOT / 'metadata' / 'flow_htbranch_internal_branch_summary.csv',
    'pinn_surrogate': OUT_ROOT / 'full' / 'pinntraj_surrogate_split_metrics.csv',
    'loss': OUT_ROOT / 'full' / f'{CONDITION_SET}_loss_history.csv',
    'test_metrics': OUT_ROOT / 'full' / f'{CONDITION_SET}_test_metrics.csv',
    'physical': OUT_ROOT / 'full' / f'{CONDITION_SET}_generated_physical_summary.csv',
    'validation': OUT_ROOT / 'full' / f'simulator_validation_{CONDITION_SET}' / 'csv' / f'{CONDITION_SET}_generated_samples_simulator_validation_summary.csv',
    'diversity': OUT_ROOT / 'full' / f'diversity_audit_{CONDITION_SET}' / 'csv' / 'diversity_decision_summary.csv',
}
for name, path in paths.items():
    print('\n##', name, path, 'exists=', path.exists())
    if path.exists() and path.suffix == '.csv':
        display(pd.read_csv(path).head(12))